# Phase 6b Stage 2 — train the two ablation finalists (Kaggle)

Trains `w125` (P3/P4/P5, width 0.125) and `p34` (P3/P4, width 0.25) from random init and compares accuracy — the piece Phase 6a deliberately left undone (it measured flash/RAM cost only, no training).

**Moved here from Colab**: free-tier Colab GPU quota couldn't absorb two 40-epoch runs plus the ~874MB dataset transfer in one sitting. Kaggle's free tier gives 30 GPU-hours/week (vs Colab's rolling, less predictable cap) and — more importantly — the ShanghaiTech dataset is already hosted on Kaggle, so it attaches to the notebook directly via **Add Data** instead of downloading through the Kaggle API. No `kaggle.json` upload needed here.

**Epoch budget cut to 20** (from 40) with `--patience 8` (40% of the cap, matching the ratio used for the killed local run's 15/40). Both variants trained comfortably past their early gains within the first ~10-15 epochs in the partial local run, so 20 epochs + early stopping should still land near the accuracy plateau without burning the whole weekly quota on one experiment.

**Update after the actual Colab run**: the Colab attempt wasn't killed by epoch count so much as by speed — its training log (recovered before cleanup) showed **~38 min/epoch** on the free T4, driven by `Using 2 dataloader workers` (CPU-bound image decode, GPU mostly idle), vs. ~5.5 min/epoch locally on the M4 Pro. At that rate even 20 epochs x 2 variants is a ~25 GPU-hour ask, well past a single free session. So this notebook now sets **`--cache disk`**: it decodes each image to a local `.npy` file once and reads that on later epochs, removing the repeated-decode cost from the hot path. Unlike `--cache ram`, this never touches system/GPU RAM — the M4 Pro hang was specifically about unified memory, which doesn't apply here either way. The cache does land on local disk (~19GB decoded) under `/kaggle/working`, which is worth watching if the run errors on disk space; if so, drop back to no `--cache` and cut the epoch budget further.

**Before running:**
1. Notebook settings (right sidebar) -> **Accelerator** -> GPU T4 x2 or P100.
2. **Internet** -> On (needed for `git clone` and `pip install`).
3. **Add Data** -> search `nikanvasei/shanghaitech-campus-dataset-test` -> Add. It will mount read-only under `/kaggle/input/`.

Then run cells top to bottom.

## 1. Clone the repo and install dependencies

In [18]:
!rm -rf /kaggle/working/edgeAi
%cd /kaggle/working
!git clone -b phase/six https://github.com/akash-reddy-k/edgeAi.git
%cd edgeAi
!git log --oneline -3
!pwd && ls

/kaggle/working
Cloning into 'edgeAi'...
remote: Enumerating objects: 130, done.
remote: Counting objects: 100% (130/130), done. (84/130)
remote: Compressing objects: 100% (104/104), done.
remote: Total 130 (delta 42), reused 69 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (130/130), 13.73 MiB | 49.17 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/kaggle/working/edgeAi
8c6c63d (HEAD -> phase/six, origin/phase/six) Kaggle Notebook | kaggle-phase-6-edgeai | Version 3
530905c Kaggle Notebook | kaggle-phase-6-edgeai | Version 2
6245cb2 Kaggle notebook: use --cache disk to fix the real Colab bottleneck
/kaggle/working/edgeAi
ablate.py     edgeAi.ipynb  kaggle		quantize.py	  train.py
benchmark.py  edgeAi.py     mcu_common.py	Readme.md
colab	      eval.py	    mcu_profile.py	requirements.txt
configs       files	    prepare_dataset.py	results


In [19]:
%cd /kaggle/working/edgeAi
!pwd && ls
!pip install -q -r requirements.txt

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — set Accelerator > GPU before continuing")

/kaggle/working/edgeAi
/kaggle/working/edgeAi
ablate.py     edgeAi.ipynb  kaggle		quantize.py	  train.py
benchmark.py  edgeAi.py     mcu_common.py	Readme.md
colab	      eval.py	    mcu_profile.py	requirements.txt
configs       files	    prepare_dataset.py	results
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 74.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.5/244.5 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 5.2 MB/s eta 0:00:00
CUDA available: True
Device: Tesla T4


## 2. Locate the attached dataset

No download needed — Kaggle mounted it read-only under `/kaggle/input/` when you added it via **Add Data**. This cell finds `frames/` and `SHANGHAI_test.txt` wherever the dataset owner's zip structure put them (same defensive search the Colab notebook uses for the API download, since Kaggle "Add Data" and `kaggle datasets download --unzip` can both nest one level deep) and **copies** — not moves, `/kaggle/input` is read-only — them into the layout `prepare_dataset.py` expects.

In [20]:
import shutil
from pathlib import Path

base = Path("data/SHANGHAI_Test")
base.mkdir(parents=True, exist_ok=True)
kaggle_input = Path("/kaggle/input")

assert kaggle_input.exists() and any(kaggle_input.iterdir()), (
    "Nothing under /kaggle/input — add the dataset via the Add Data panel "
    "(search nikanvasei/shanghaitech-campus-dataset-test) before running this cell."
)

if not (base / "frames").exists():
    candidates = [p for p in kaggle_input.rglob("frames") if p.is_dir()]
    assert candidates, "No frames/ directory found under /kaggle/input — inspect the attached dataset manually"
    shutil.copytree(candidates[0], base / "frames")

if not (base / "SHANGHAI_test.txt").exists():
    candidates = list(kaggle_input.rglob("SHANGHAI_test.txt"))
    assert candidates, "No SHANGHAI_test.txt found under /kaggle/input — inspect the attached dataset manually"
    shutil.copy(candidates[0], base / "SHANGHAI_test.txt")

n_scenes = len(list((base / "frames").iterdir()))
print(f"OK — {n_scenes} scenes under {base}/frames")

OK — 199 scenes under data/SHANGHAI_Test/frames


## 3. Base weights + pseudo-labeled dataset

`prepare_dataset.py` uses stock `yolov8n.pt` as a pseudo-labeler (COCO -> campus domain adaptation), same as Phase 3 locally.

In [21]:
import os, shutil
from ultralytics import YOLO

os.makedirs("models", exist_ok=True)
YOLO("yolov8n.pt")  # downloads to cwd if not already cached
if os.path.exists("yolov8n.pt") and not os.path.exists("models/yolov8n.pt"):
    shutil.move("yolov8n.pt", "models/yolov8n.pt")
print("models/yolov8n.pt present:", os.path.exists("models/yolov8n.pt"))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
models/yolov8n.pt present: True


In [22]:
!python prepare_dataset.py

Scenes found: 199  |  Val scenes: 8

  [train] 01_001         765 frames  detections: 1249
  [train] 01_0015        433 frames  detections: 1153
  [train] 01_0025        601 frames  detections: 2226
  [train] 01_0027        409 frames  detections: 1944
  [val  ] 01_0028        457 frames  detections: 1248
  [train] 01_004         770 frames  detections: 2708
  [train] 01_0051        337 frames  detections: 841
  [train] 01_0052        337 frames  detections: 3188
  [train] 01_0053        457 frames  detections: 3052
  [val  ] 01_0055        313 frames  detections: 1329
  [train] 01_0056        529 frames  detections: 1957
  [val  ] 01_006         506 frames  detections: 571
  [train] 01_0064        289 frames  detections: 1821
  [train] 01_008        1010 frames  detections: 3025
  [train] 01_012         738 frames  detections: 1380
  [train] 01_0135        409 frames  detections: 550
  [train] 01_0136        529 frames  detections: 2654
  [val  ] 01_0139        217 frames  detections:

## 4. Train the two Stage 2 finalists

Both from random init (`--variant` implies `pretrained=False` in `train.py` — see its docstring for why). `--device 0` targets the Kaggle GPU. `--cache disk`: decode-once-then-reuse, see the caveat at the top for why (not `--cache ram` — that's the one that's unsafe, and only on unified-memory machines anyway). `--epochs 20 --patience 8`: cut from the original 40/15 budget to fit Kaggle's weekly GPU quota across both runs.

In [ ]:
!python train.py --variant w125 --epochs 20 --patience 8 --batch 16 --device 0 --cache disk

Phase 6b Stage 2 — training P3/P4/P5  w0.125  (key=w125)  from random init
  epochs=20  patience=8  batch=16  imgsz=640  device=0

Ultralytics 8.4.171 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=disk, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/yolo_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=trai

In [ ]:
!python train.py --variant p34 --epochs 20 --patience 8 --batch 16 --device 0 --cache disk

## 5. Package results for download

Zips both run directories (`results.csv`, `args.yaml`, plots) and the best weights into `/kaggle/working/stage2_results.zip`. Kaggle persists everything under `/kaggle/working/` as the notebook's **Output** — after the run finishes (or you Save Version), download it from the Output tab in the right sidebar (no `files.download()` here; that's a Colab-only API). Unzip locally into the same paths — `training_runs/*` -> `runs/detect/results/training/`, `models/*.pt` -> `models/` — and everything downstream (the Stage 2 comparison, README) reads it exactly like a local run.

In [ ]:
import shutil, os

stage_dir = "/kaggle/working/stage2_package"
os.makedirs(f"{stage_dir}/models", exist_ok=True)
shutil.copytree("runs/detect/results/training", f"{stage_dir}/training_runs", dirs_exist_ok=True)

for f in ["models/yolov8_ablation_w125.pt", "models/yolov8_ablation_p34.pt"]:
    if os.path.exists(f):
        shutil.copy(f, f"{stage_dir}/models/")
    else:
        print(f"WARNING: {f} missing — training may not have completed")

shutil.make_archive("/kaggle/working/stage2_results", "zip", stage_dir)
print("Wrote /kaggle/working/stage2_results.zip — download it from the Output tab.")

In [ ]:
# Quick summary printed here too, in case you'd rather not download and unzip —
# paste this back into the conversation.
import pandas as pd

for variant in ["w125", "p34"]:
    csv_path = f"runs/detect/results/training/yolov8_ablation_{variant}/results.csv"
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df.columns = [c.strip() for c in df.columns]
        last = df.iloc[-1]
        print(f"--- {variant} ---")
        print(f"  epochs run : {len(df)}")
        print(f"  mAP50      : {last['metrics/mAP50(B)']:.4f}")
        print(f"  mAP50-95   : {last['metrics/mAP50-95(B)']:.4f}")
        print(f"  Precision  : {last['metrics/precision(B)']:.4f}")
        print(f"  Recall     : {last['metrics/recall(B)']:.4f}\n")